# Example 0: Understanding Decision Tree Classifiers with the Iris Dataset


| Iris setosa | Iris versicolor | Iris virginica |
|:-------------------------------------:|:-------------------------------------:|:-------------------------------------:|
| ![](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Irissetosa1.jpg/330px-Irissetosa1.jpg) | ![](https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/Blue_Flag%2C_Ottawa.jpg/250px-Blue_Flag%2C_Ottawa.jpg) | ![](https://upload.wikimedia.org/wikipedia/commons/thumb/f/f8/Iris_virginica_2.jpg/250px-Iris_virginica_2.jpg) |

## References
- https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
- https://en.wikipedia.org/wiki/Iris_flower_data_set

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import seaborn as sns
import pandas as pd

## 1. Load the iris dataset

In [2]:
dataset = load_iris(as_frame=True)

In [3]:
print(dataset.keys())

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])


In [4]:
print(dataset.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [5]:
dataset.data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [6]:
print(dataset.target_names)

['setosa' 'versicolor' 'virginica']


In [7]:
dataset.target.head()

0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

In [ ]:
# Pairplot
df = dataset.data.copy()
df['species'] = pd.Categorical.from_codes(dataset.target, dataset.target_names)
sns.pairplot(df, hue='species', corner=True)

In [9]:
# Convert pandas.DataFrame to numpy.ndarray
x = dataset.data.to_numpy()  # feature matrix
y = dataset.target.to_numpy()  # target labels

# Split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

# 2. Create and train a Decision Tree classifier

In [10]:
model = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)
model.fit(x_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)

# 3. Make predictions and evaluate accuracy

In [11]:
y_pred = model.predict(x_test)
acc = accuracy_score(y_true=y_test, y_pred=y_pred)
print(f"Test accuracy: {acc:.2f}")

Test accuracy: 0.97


# 4. Visualize the decision tree

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(model, filled=True, feature_names=dataset.feature_names, class_names=dataset.target_names)
plt.title("Decision Tree for Iris Dataset")

In [13]:
y_pred

array([1, 0, 2, 1, 2, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2,
       0, 2, 2, 2, 2, 2, 0, 0])

# Find indices of top-2 important features

In [14]:
for name, importance in zip(dataset.feature_names, model.feature_importances_):
    print(f'{name}: {importance:.3f}')

sepal length (cm): 0.000
sepal width (cm): 0.000
petal length (cm): 0.957
petal width (cm): 0.043


In [15]:
top2_idx = np.argsort(model.feature_importances_)[-2:]

In [16]:
# Create 2D meshgrid over the top-2 features
f1, f2 = top2_idx
x_min, x_max = x_test[:, f1].min() - 1, x_test[:, f1].max() + 1
y_min, y_max = x_test[:, f2].min() - 1, x_test[:, f2].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

In [17]:
x_vis = x
y_vis = y

In [ ]:
# Fix other features to their mean values
x_fixed = np.mean(x_test, axis=0)
grid = np.zeros((xx.size, x.shape[1])) + x_fixed  # broadcast fixed values
grid[:, f1] = xx.ravel()
grid[:, f2] = yy.ravel()

# Predict and plot
z = model.predict(grid).reshape(xx.shape)
fig, ax = plt.subplots()
contourf = ax.contourf(xx, yy, z, alpha=0.4, cmap=plt.cm.RdYlBu)

for label, (color, marker) in enumerate(zip(['crimson', 'gold', 'cornflowerblue'], ['o', '^', 's'])):
    ax.scatter(x_vis[:, f1][y_vis == label], x_vis[:, f2][y_vis == label], color=color, edgecolor='black', marker=marker, label=dataset.target_names[label])
ax.legend()
ax.set_xlabel(dataset.feature_names[f1])
ax.set_ylabel(dataset.feature_names[f2])
ax.set_title("Decision Boundary (Top-2 Important Features)")